# Day 2 — PyTorch & Autograd Sanity Checks

## 1. Tensor Basics

In [74]:
import torch
import numpy as np

print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cpu
False


$$
\begin{aligned}
x_\text{rand} &= \texttt{torch.rand}(x.\text{shape}) \quad \longleftrightarrow \quad \texttt{np.random.rand}(*x.\text{shape}) \\
&\text{在shape的范围内，随机生成0~1之间的数填充。}
\end{aligned}
$$

In [75]:
data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)

x_ones = torch.ones_like(x_data) # retains the properties of x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # overrides the datatype of x_data
print(f"Random Tensor: \n {x_rand} \n")

Ones Tensor: 
 tensor([[1, 1],
        [1, 1]]) 

Random Tensor: 
 tensor([[0.4270, 0.3594],
        [0.3392, 0.3555]]) 



#### 矩阵乘法
$$

@ <-----> matmul \\
y=tensor @ tensor.T <-----> y=tensor.matmul(tensor.T) <-----> torch.matmul(tensor,tensor.T,out=y) 

$$
#### 算数乘法
$$

* <-----> mul \\
y=tensor * tensor.T <-----> y=tensor.mul(tensor.T) <-----> torch.mul(tensor,tensor.T,out=y) \\
最后输出到out对应的y。
$$


In [76]:
X = torch.randn(4, 3)
W = torch.randn(3, 2)

Y = X @ W

print(X.shape)
print(W.shape)
print(Y.shape)

print(X)

torch.Size([4, 3])
torch.Size([3, 2])
torch.Size([4, 2])
tensor([[-0.3142,  0.7270,  0.6438],
        [-0.7127,  0.8403, -0.2345],
        [ 0.4528, -1.3745,  1.3069],
        [ 1.5700,  1.8817,  0.8897]])


In [77]:
X.sum(dim=0)
X.sum(dim=1)

X.mean(dim=0)
X.mean(dim=1)

tensor([ 0.3522, -0.0356,  0.1284,  1.4471])

## Bridge with NumPy

In [78]:
a = np.array([1., 2., 3.])

t = torch.from_numpy(a)

print(a)
print(t)

a[0]=100
print(a)
print(t)

[1. 2. 3.]
tensor([1., 2., 3.], dtype=torch.float64)
[100.   2.   3.]
tensor([100.,   2.,   3.], dtype=torch.float64)


In [79]:
t = torch.tensor([1., 2., 3.])
a = t.numpy()

print(a)
print(t)

t[0]=100
print(a)
print(t)

[1. 2. 3.]
tensor([1., 2., 3.])
[100.   2.   3.]
tensor([100.,   2.,   3.])


### 维度推断
$$
pytorch:  \\
y=x.view(-1,3)  \\
y=x.view(-1,3)  \\
numpy:   \\
 y=x.reshape(-1,3)
$$

In [80]:
x=torch.tensor([[1. ,2. ,3.],[4. ,5. ,6. ]])
y=x.view(-1,2)
print(x)
print(y)

z=x.reshape((3,2))
print(z)
z=y.reshape(-1,3)
print(z)

tensor([[1., 2., 3.],
        [4., 5., 6.]])
tensor([[1., 2.],
        [3., 4.],
        [5., 6.]])
tensor([[1., 2.],
        [3., 4.],
        [5., 6.]])
tensor([[1., 2., 3.],
        [4., 5., 6.]])


### 返回元素总数
$$
pytorch: number=x.numel() numel为 \, number \, of \, element的缩写 \\
numpy: number=x.size()
$$

### automatic differentiation / Autograd


In [81]:
x = torch.tensor(
    3.0,
    requires_grad=True
)

y = x ** 2

y.backward()

print(x.grad)

tensor(6.)


$$
y.backward()触发反向传播（backpropagation）,Autograd 顺着计算图，\\
链式法则求导，算出 dxdy​,结果会存到 x.grad 属性里面
$$

## 2. Stable Softmax in PyTorch

In [82]:
import torch

logits = torch.tensor([
    [1000., 1001., 1002.],
    [2000., 1998., 2001.]
])

print(logits)
print(logits.shape)

tensor([[1000., 1001., 1002.],
        [2000., 1998., 2001.]])
torch.Size([2, 3])


In [83]:
max_logits=logits.max(dim=1,keepdim=True).values
new_logits=logits-max_logits
exp=torch.exp(new_logits)
softmax=exp/exp.sum(dim=1,keepdim=True)
print(softmax)
print(softmax.sum(dim=1))

tensor([[0.0900, 0.2447, 0.6652],
        [0.2595, 0.0351, 0.7054]])
tensor([1., 1.])


##  3.Cross Entropy

In [84]:
labels = torch.tensor([2, 0])
correct_probs=softmax[torch.arange(len(labels)),labels]
print(correct_probs)
losses=-torch.log(correct_probs)
loss=torch.sum(losses)/len(losses)
print(loss)
print(loss.shape)

tensor([0.6652, 0.2595])
tensor(0.8783)
torch.Size([])


In [85]:
import torch.nn.functional as F

official_loss = F.cross_entropy(logits, labels)

print("our loss:     ", loss)
print("PyTorch loss: ", official_loss)
print("difference:   ", torch.abs(loss - official_loss))

our loss:      tensor(0.8783)
PyTorch loss:  tensor(0.8783)
difference:    tensor(0.)


## 4. Autograd Check 1 — y = x²

In [86]:
x = torch.tensor(
    3.0,
    requires_grad=True
)

y = x ** 2

print("x =", x)
print("y =", y)

print("x.requires_grad =", x.requires_grad)
print("y.requires_grad =", y.requires_grad)

print("x.grad =", x.grad)
print("y.grad_fn =", y.grad_fn)
#----------------------------------------------------
y.backward()

print(x.grad)



x = tensor(3., requires_grad=True)
y = tensor(9., grad_fn=<PowBackward0>)
x.requires_grad = True
y.requires_grad = True
x.grad = None
y.grad_fn = <PowBackward0 object at 0x000002251BE27E80>
tensor(6.)


## 5. Autograd Check 2 — Chain Rule

In [87]:
x=torch.tensor(
    3.0,
    requires_grad=True
)
u=2*x+1
y=u**2
print(u.grad_fn)
print(y.grad_fn)

y.backward()
print(x.grad)
print(u.grad)
print(y.grad)


tensor(28.)
None
None


C:\Users\22807\AppData\Local\Temp\ipykernel_37300\460750581.py:12: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:499.)
  print(u.grad)
C:\Users\22807\AppData\Local\Temp\ipykernel_37300\460750581.py:13: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Te

## 6. Autograd Check3—从 Autograd 到模型参数

In [88]:
x=torch.tensor(2)
w=torch.tensor(
    3.0,
    requires_grad=True
)
b=torch.tensor(
    1.0,
    requires_grad=True    
)
y=torch.tensor(10.0)
y1=w*x+b
loss=(y1-y)**2
loss.backward()
print(loss)
print(w.grad)
print(b.grad)


tensor(9., grad_fn=<PowBackward0>)
tensor(-12.)
tensor(-6.)


$$
假设学习率lr=0.1 \\
则计算得新的w=4.2，b=1.6,重新做新的forward,观察loss
$$

In [89]:
x=torch.tensor(2)
w=torch.tensor(
    4.2,
    requires_grad=True
)
b=torch.tensor(
    1.6,
    requires_grad=True    
)
y=torch.tensor(10.0)
y1=w*x+b
loss=(y1-y)**2
loss.backward()
print(loss)
print(w.grad)
print(b.grad)


tensor(0., grad_fn=<PowBackward0>)
tensor(0.)
tensor(0.)


$$
如果换成自动更新时，要进行梯度禁用，with torch.no_grad():\\
梯度会累积，要及时清零
$$

In [90]:
w = torch.tensor(3.0, requires_grad=True)

loss = w ** 2
loss.backward()
print(w.grad)

loss = w ** 2
loss.backward()
print(w.grad)

tensor(6.)
tensor(12.)


## Step 4 — nn.Module，把“参数”变成真正的模型

In [91]:
import os
import torch
from torch import nn
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [92]:
X = torch.rand(1, 28, 28)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
print(pred_probab)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

tensor([[0.0994, 0.1013, 0.0947, 0.1000, 0.1016, 0.1025, 0.0998, 0.1040, 0.0972,
         0.0996]], grad_fn=<SoftmaxBackward0>)
Predicted class: tensor([7])


## 7. My First nn.Module — Linear Model

In [108]:
import torch
from torch import nn

class MyLinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.w=nn.Parameter(torch.tensor(3.0))
        self.b=nn.Parameter(torch.tensor(1.0))

    def forward(self,x):
        y_hat=self.w*x+self.b
        return y_hat

model = MyLinearModel()

x = torch.tensor(2.0)
y = torch.tensor(10.0)

loss_fn = nn.MSELoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

for epoch in range(10):
    optimizer.zero_grad()
    y_hat=model(x)
    loss=loss_fn(y,y_hat)
    loss.backward()
    optimizer.step()
    print(
        epoch,
        loss.item(),
        model.w.item(),
        model.b.item()
    )
    
        

0 9.0 3.119999885559082 1.059999942779541
1 7.290001392364502 3.2279999256134033 1.1139999628067017
2 5.904901504516602 3.325199842453003 1.1625999212265015
3 4.782970428466797 3.41267991065979 1.206339955329895
4 3.874204397201538 3.4914119243621826 1.2457059621810913
5 3.138106107711792 3.5622706413269043 1.2811353206634521
6 2.5418663024902344 3.6260435581207275 1.3130217790603638
7 2.0589115619659424 3.683439254760742 1.341719627380371
8 1.667718768119812 3.735095262527466 1.367547631263733
9 1.3508528470993042 3.781585693359375 1.3907928466796875


- loss_fn = nn.MSELoss()：实例化损失函数，不计算损失；调用 loss_fn(y_pred, y_true) 才计算loss
- optimizer = torch.optim.SGD(参数, lr)：实例化SGD优化器，不更新参数
- SGD：随机梯度下降，利用梯度更新权重
- optimizer.step()：执行参数更新（optimizer真正被调用）
- optimizer.zero_grad()：清空历史梯度

In [109]:
x = torch.tensor([
    [1.0],
    [2.0],
    [3.0],
    [4.0]
])

y = torch.tensor([
    [3.0],
    [5.0],
    [7.0],
    [9.0]
])
model = nn.Linear(1, 1)
print(model.weight)
print(model.bias)

print(list(model.parameters()))

y_hat = model(x)

print(y_hat)
print(y_hat.shape)

Parameter containing:
tensor([[-0.0553]], requires_grad=True)
Parameter containing:
tensor([0.9473], requires_grad=True)
[Parameter containing:
tensor([[-0.0553]], requires_grad=True), Parameter containing:
tensor([0.9473], requires_grad=True)]
tensor([[0.8920],
        [0.8368],
        [0.7815],
        [0.7263]], grad_fn=<AddmmBackward0>)
torch.Size([4, 1])


nn.Linear(1, 1) 中两个 1 分别表示：

in_features  = 1
out_features = 1

而：

x.shape = (4, 1)
          ↑  ↑
       4个样本 每个样本1个feature

nn.Linear(1,1) 对最后一个维度做线性变换：

(4,1​)→(4,1)

In [10]:
import torch
from torch import nn
x = torch.tensor([
    [1.0],
    [2.0],
    [3.0],
    [4.0]
])

y = torch.tensor([
    [3.0],
    [5.0],
    [7.0],
    [9.0]
])

model=nn.Linear(1,1)
loss_fn=nn.MSELoss()
optimizer=torch.optim.SGD(
    model.parameters(),
    lr=0.01
)
for epoch in range(100):
    optimizer.zero_grad()
    y_hat=model(x)
    loss=loss_fn(y,y_hat)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 99:
        print(
            epoch,
            loss.item(),
            model.weight.item(),
            model.bias.item()
        )
print("Final weight:", model.weight.item())
print("Final bias:", model.bias.item())


99 4.0915496356319636e-05 2.005307674407959 0.9843949675559998
Final weight: 2.005307674407959
Final bias: 0.9843949675559998


In [12]:
import torch
from torch import nn

model = nn.Sequential(
    nn.Linear(4, 4),
    nn.Dropout(p=0.5)
)

x = torch.ones(1, 4)

model.train()

print(model(x))
print(model(x))
print(model(x))

model.eval()

print(model(x))
print(model(x))
print(model(x))

tensor([[-0.0000, -0.0000, -0.0000, -2.0187]], grad_fn=<MulBackward0>)
tensor([[-0.0000, -0.6035, -2.5929, -0.0000]], grad_fn=<MulBackward0>)
tensor([[-0.0000, -0.6035, -2.5929, -0.0000]], grad_fn=<MulBackward0>)
tensor([[-0.9005, -0.3018, -1.2964, -1.0093]], grad_fn=<AddmmBackward0>)
tensor([[-0.9005, -0.3018, -1.2964, -1.0093]], grad_fn=<AddmmBackward0>)
tensor([[-0.9005, -0.3018, -1.2964, -1.0093]], grad_fn=<AddmmBackward0>)


### MLP综合练习

In [16]:
import torch
from torch import nn
class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1=nn.Linear(4,8)
        self.relu=nn.ReLU()
        self.fc2=nn.Linear(8,3)

    def forward(self,x):
        x=self.fc1(x)
        x=self.relu(x)
        logits=self.fc2(x)
        return logits

model=TinyClassifier()
x=torch.randn(5,4)
logits=model(x)


print("input shape:", x.shape)
print("logits shape:", logits.shape)

print(model)


labels=torch.tensor([0,2,1,0,2])
loss_fn=nn.CrossEntropyLoss()

loss=loss_fn(logits,labels)
print("loss:",loss)

optimizer=torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)
optimizer.zero_grad()
loss.backward()
print(model.fc1.weight.grad.shape)
optimizer.step()



input shape: torch.Size([5, 4])
logits shape: torch.Size([5, 3])
TinyClassifier(
  (fc1): Linear(in_features=4, out_features=8, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=8, out_features=3, bias=True)
)
loss: tensor(1.1280, grad_fn=<NllLossBackward0>)
torch.Size([8, 4])
